# ODI to Databricks Migration: TRG_EMP_LOAD

**Source File**: TARGET.txt
**Conversion Date**: 2024-07-30
**Description**: Loads data into the TRG_EMP table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "FULL_LOAD", "1. ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "101", "2. Datasource Num ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "3. ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "4. ODI Session Number")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", "2024-01-01 00:00:00", "5. Current Extract Time")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "6. Last Extract Time")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS
SELECT '${ETL_JOB_TYPE}' AS etl_job_type;

CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS
SELECT ${DATASOURCE_NUM_ID} AS datasource_num_id;

CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS
SELECT ${ETL_PROC_WID} AS etl_proc_wid;

CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS
SELECT '${ODI_SESS_NO}' AS odi_sess_no;

CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time;

In [ ]:
display(spark.sql("""
  SELECT * FROM v_etl_job_type
  UNION ALL
  SELECT * FROM v_datasource_num_id
  UNION ALL
  SELECT * FROM v_etl_proc_wid
  UNION ALL
  SELECT * FROM v_odi_sess_no
  UNION ALL
  SELECT * FROM v_etl_current_extract_time
  UNION ALL
  SELECT * FROM v_etl_last_extract_time
"""))

## Insert Data

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
INSERT 
  INTO workspace.hr.trg_emp
  (
    employee_id ,
    first_name ,
    last_name ,
    email ,
    phone_number ,
    hire_date ,
    job_id ,
    salary ,
    commission_pct ,
    manager_id ,
    department_id 
  ) 
SELECT 
  EMPLOYEES.EMPLOYEE_ID ,
  EMPLOYEES.FIRST_NAME ,
  EMPLOYEES.LAST_NAME ,
  EMPLOYEES.EMAIL ,
  EMPLOYEES.PHONE_NUMBER ,
  EMPLOYEES.HIRE_DATE ,
  EMPLOYEES.JOB_ID ,
  EMPLOYEES.SALARY ,
  EMPLOYEES.COMMISSION_PCT ,
  EMPLOYEES.MANAGER_ID ,
  EMPLOYEES.DEPARTMENT_ID  
FROM 
  workspace.hr.employees EMPLOYEES;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS target_record_count FROM workspace.hr.trg_emp;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Names**: Oracle schema `HR` converted to `workspace.hr`. Table names `TRG_EMP` and `EMPLOYEES` converted to lowercase `trg_emp` and `employees`.
2.  **Oracle Hints**: The `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable in Databricks Spark SQL.
3.  **Target Table DDL**: The DDL for `workspace.hr.trg_emp` is not provided in the source file. Ensure this table exists with compatible data types before running this notebook.
4.  **ETL Parameters**: Standard ETL widgets are created and made available as temporary views (`v_etl_job_type`, etc.) for potential use in more complex ETL flows. Although not directly used by this simple INSERT statement, they follow the standard notebook template.